# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*




### Unit of Analysis
One row represents **one webpage for one month**. Each row contains SEO and content metrics such as impressions, clicks, CTR, average position, and content-related information for that webpage.

### Time Window
For this assignment, I use the **March 2026 (2026-03)** data because it is a mid-panel month. This avoids using the final month (June 2026), which should be reserved as a test period.

In [4]:
import pandas as pd
from pathlib import Path

# Find project root
current = Path.cwd()
while not (current / "data").exists():
    if current.parent == current:
        raise FileNotFoundError("Could not find the data folder.")
    current = current.parent

# Load dataset
data_path = current / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print(df.head())

             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-25000  0.76 

### Conclusion

The query confirms that each row represents one webpage for one month. The analysis uses the March 2026 data slice, which is appropriate for feature engineering because it is a mid-panel month and avoids information from the final outcome month.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



| Category | Fields | Why |
|----------|--------|-----|
| **Features** | `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr` | These are available before making a prediction and are used as model inputs. |
| **Label** | `is_declining_label` (derived from `trend_direction`) | This is the target the model predicts: whether a webpage is declining (1) or not (0). |
| **Context** | `page_url`, `content_type`, `month` | These fields provide information about the webpage and dataset but are not directly used for prediction. |
| **Excluded** | `trend_direction` | Excluded because it is used to create the target label. Including it would cause **data leakage**, allowing the model to indirectly see the answer before prediction. |

In [5]:
# Display all available columns in the dataset
print("Columns in the dataset:\n")
for col in df.columns:
    print("-", col)

Columns in the dataset:

- content_id
- client_id
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier
- trend_direction
- trend_pct


In [6]:
# Example (change column names if needed)
df[[
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "trend_direction"
]].head()

,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction
0,187,20,3803,10.6,0.76,down
1,445,25,15320,20.3,0.05,down
2,141,20,12581,36.5,0.09,down
3,463,22,11751,6.2,0.49,stable
4,263,14,19140,44.0,0.13,down


### Conclusion

The fields have been separated into features, labels, context, and excluded columns. Only information available before prediction is used as features. The label and any label-derived fields are excluded from the feature set to prevent data leakage and ensure an honest machine learning workflow.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# Display first 5 rows to verify the unit of analysis
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


The output shows that each row represents one webpage with its associated SEO and content metrics.

In [9]:
print("Total Rows:", len(df))
print(df.describe())

Total Rows: 30000
       search_volume   competition           cpc    word_count     char_count  \
count   27532.000000  27532.000000  27532.000000  22301.000000   22301.000000   
mean      158.882391      0.146954      0.485342   3107.760325   20665.277835   
std      1518.270825      0.285241      2.101560   1452.382598   10115.344042   
min         0.000000      0.000000      0.000000      8.000000      40.000000   
25%         0.000000      0.000000      0.000000   2413.000000   15644.000000   
50%        10.000000      0.000000      0.000000   2877.000000   19116.000000   
75%        20.000000      0.130000      0.000000   3666.000000   24011.000000   
max     74000.000000      1.000000    100.360000   9546.000000  111158.000000   

       impressions_90d    clicks_90d  pageviews_90d  sessions_90d  \
count     30000.000000  30000.000000   30000.000000  30000.000000   
mean       5200.366300     16.097333      49.942467     37.066633   
std       16838.019547     75.076958     152.

The dataset contains the expected number of records. The selected month (March 2026) is used for analysis.

In [10]:
missing = df.isnull().sum()

missing[missing > 0]

search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64

The query identifies which columns contain missing values so they can be handled before modeling.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



This dataset has several limitations that should be considered before building or evaluating a machine learning model:

1. **Limited Time Window**
   - This analysis uses a single monthly snapshot (March 2026), so it may not capture seasonal or long-term trends.

2. **Historical Data Only**
   - The model only uses historical SEO and content metrics. It cannot predict unexpected future events such as algorithm updates or sudden changes in user behavior.

3. **Potential Missing Values**
   - Some features may contain missing or incomplete values, which can affect model performance if not handled properly.

4. **No Direct Measure of Content Quality**
   - The dataset contains SEO metrics but does not directly measure content quality, readability, or user satisfaction.

5. **Risk of Data Leakage**
   - Any feature derived from the target label (such as `trend_direction`) must be excluded from the feature set to avoid unrealistically high model performance.

### Conclusion

The dataset is suitable for learning and feature engineering, but it has limitations related to historical coverage, missing information, and the possibility of data leakage. Understanding these constraints helps build a more reliable and realistic machine learning model.

# Self-check

Before submitting, I verified the following:

- [x] Every section above is filled with both markdown explanations and supporting code where required.
- [x] The notebook runs from top to bottom without errors.
- [x] No client names, URLs, private queries, or sensitive information are included.
- [x] My conclusions use careful wording such as *observed*, *measured*, *directional*, and *decision-support*.
- [x] The notebook has been committed to my GitHub repository under `work/notebooks/`.